# 🏠 Full Floor Plan Generation Pipeline

This notebook implements the **complete end-to-end pipeline**:

1. **Input**: Wall/boundary images (from ZIMHE dataset or custom)
2. **CNN Segmentation**: Extract room masks using DeepLabV3+
3. **LoRA Inference**: Generate furnished rooms per room type using FLUX.1-dev
4. **Integration**: Combine generated rooms into final floor plan

---

## 📦 Step 0: Setup & Imports

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print(f'✓ floor_planrules path added: {FLOOR_PLANRULES_PATH}')

# Add floor_planrules to path for integration    sys.path.insert(0, FLOOR_PLANRULES_PATH)

import sysif FLOOR_PLANRULES_PATH not in sys.path:
FLOOR_PLANRULES_PATH = '/content/drive/MyDrive/floor_planrules'

In [ ]:
# Install required packages
!pip install -q diffusers transformers accelerate safetensors
!pip install -q opencv-python albumentations
!pip install -q inference_sdk shapely
!pip install -q controlnet-aux
print("✓ All packages installed")

In [ ]:
import os
import cv2
import numpy as np
import torch
import torchvision.transforms as T
from torchvision.models.segmentation import deeplabv3_resnet50
from PIL import Image
import matplotlib.pyplot as plt
from glob import glob
import json
import random
from tqdm.notebook import tqdm
from typing import List, Dict, Tuple
from inference_sdk import InferenceHTTPClient
from shapely.geometry import Polygon, Point

from shapely.affinity import rotate as shapely_rotate, translate as shapely_translateprint("✓ Integration modules imported")

print(f"Using device: {device}")

# Import floor_planrules modulesdevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from room_analyzer import extract_room_dimensions, color_maps, room_types# Set device

from furniture_definitions import load_furniture_prototypes, FURNITURE_ROOM_MAP, Furniture
from furniture_placer import FurniturePlacer

## 🔧 Configuration

In [ ]:
# ============================================================
# CONFIGURATION - Modify these paths as needed
# ============================================================

# Input: Wall images folder
WALL_IMAGES_DIR = '/content/drive/MyDrive/cnn_dataset/walls'  # or your ZIMHE dataset path

# CNN Segmentation model
CNN_MODEL_PATH = '/content/drive/MyDrive/cnn_dataset/best_model_final_4.pth'

# Output directories
SEGMENTATION_OUTPUT = '/content/drive/MyDrive/pipeline_output/segmentation'
EXTRACTION_OUTPUT = '/content/drive/MyDrive/pipeline_output/extraction'
LORA_OUTPUT = '/content/drive/MyDrive/pipeline_output/generated'
FINAL_OUTPUT = '/content/drive/MyDrive/pipeline_output/final'

# LoRA weights paths (per room type)
LORA_WEIGHTS = {
    'living_room': '/content/drive/MyDrive/outputs_livingroom_new',
    'bedroom': '/content/drive/MyDrive/outputs_bedroom_modified',
    'bathroom': '/content/drive/MyDrive/outputs_bathroom_modified',
    'hallway': '/content/drive/MyDrive/outputs_hallways',
    'toilet': '/content/drive/MyDrive/outputs_toilet'
}

# Floor plan rules path
FLOOR_PLANRULES_PATH = '/content/drive/MyDrive/floor_planrules'
FURNITURE_JSON_PATH = '/content/drive/MyDrive/floor_planrules/furniture_crops/furniture_measurements.json'

# Roboflow API for YOLO furniture detection
ROBOFLOW_API_KEY = 'bngHGPb2UJDgzZlQGaTu'
ROBOFLOW_MODEL_ID = 'furniture-ya4pe/1'

# Create output directories
for output_dir in [SEGMENTATION_OUTPUT, EXTRACTION_OUTPUT, LORA_OUTPUT, FINAL_OUTPUT]:

    os.makedirs(output_dir, exist_ok=True)print(f"  CNN model: {CNN_MODEL_PATH}")

    print(f"  Wall images: {WALL_IMAGES_DIR}")
print("✓ Configuration loaded")

---
## 🧠 Part 1: CNN Segmentation

Load the trained DeepLabV3+ model and segment wall images into room masks.

In [ ]:
# Color map for segmentation classes
COLOR_MAP = np.array([
    [0, 0, 0],         # 0: background
    [255, 154, 0],     # 1: bedroom (orange)
    [50, 99, 155],     # 2: hallway (purple)
    [0, 155, 255],     # 3: bathroom (blue)
    [155, 255, 0],     # 4: living room (yellow-green)
    [100, 80, 71]      # 5: empty-room (brown)
], dtype=np.uint8)

CLASS_NAMES = ['background', 'bedroom', 'hallway', 'bathroom', 'living_room', 'empty_room']
NUM_CLASSES = len(CLASS_NAMES)

# Map class index to room type for LoRA
CLASS_TO_ROOM = {
    1: 'bedroom',
    2: 'hallway',
    3: 'bathroom',
    4: 'living_room',
    5: 'empty_room'  # Will use living_room LoRA as fallback
}

print(f"✓ {NUM_CLASSES} classes: {CLASS_NAMES}")

In [ ]:
def post_process(seg_map, min_area=500):
    """Remove small connected components from segmentation."""
    out = np.zeros_like(seg_map)
    unique_classes = np.unique(seg_map)

    for cls in unique_classes:
        mask = (seg_map == cls).astype(np.uint8)
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask)

        for label in range(1, num_labels):
            area = stats[label, cv2.CC_STAT_AREA]
            if area >= min_area:
                out[labels == label] = cls

    return out


def load_segmentation_model(model_path):
    """Load the trained DeepLabV3+ model."""
    model = deeplabv3_resnet50(pretrained_backbone=False, num_classes=NUM_CLASSES)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    print(f"✓ CNN model loaded from {model_path}")
    return model


def segment_image(model, image_path, img_size=(512, 512)):
    """
    Segment a wall image into room masks.
    
    Returns:
        seg_map: Segmentation map (H, W) with class indices
        color_overlay: RGB visualization of segmentation
        orig: Original image
    """
    # Load and preprocess
    orig = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    H0, W0 = orig.shape[:2]
    
    inp = cv2.resize(orig, img_size)
    inp_tensor = T.ToTensor()(inp).unsqueeze(0).to(device)
    
    # Inference
    with torch.no_grad():
        out = model(inp_tensor)['out'][0]
        prob = torch.softmax(out, dim=0).cpu().numpy()
        seg = prob.argmax(axis=0).astype(np.uint8)
        
        # Resize back to original size
        seg = cv2.resize(seg, (W0, H0), interpolation=cv2.INTER_NEAREST)
        seg = post_process(seg)
    
    # Create color overlay
    color_overlay = COLOR_MAP[seg]
    
    return seg, color_overlay, orig


# Load the model
cnn_model = load_segmentation_model(CNN_MODEL_PATH)

In [ ]:
def extract_room_masks(seg_map, original_image, output_dir, base_name):
    """
    Extract individual room masks from segmentation.
    
    Returns:
        room_masks: Dict mapping room_type -> list of (mask_path, mask_array, bbox)
    """
    room_masks = {}
    
    for class_idx in range(1, NUM_CLASSES):  # Skip background
        class_mask = (seg_map == class_idx).astype(np.uint8)
        
        if class_mask.sum() == 0:
            continue
            
        room_type = CLASS_TO_ROOM.get(class_idx, 'unknown')
        
        # Find connected components (multiple instances of same room type)
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(class_mask)
        
        for label_idx in range(1, num_labels):  # Skip background label
            instance_mask = (labels == label_idx).astype(np.uint8) * 255
            
            # Get bounding box
            x, y, w, h, area = stats[label_idx]
            
            if area < 500:  # Skip tiny regions
                continue
            
            # Create room key with instance number
            if room_type not in room_masks:
                room_masks[room_type] = []
            
            instance_num = len(room_masks[room_type]) + 1
            mask_name = f"{base_name}_{room_type}_{instance_num}.png"
            mask_path = os.path.join(output_dir, room_type, mask_name)
            
            # Save mask
            os.makedirs(os.path.dirname(mask_path), exist_ok=True)
            cv2.imwrite(mask_path, instance_mask)
            
            room_masks[room_type].append({
                'path': mask_path,
                'mask': instance_mask,
                'bbox': (x, y, w, h),
                'area': area,
                'instance': instance_num
            })
    
    return room_masks


print("✓ Room extraction functions defined")

In [ ]:
# Process a single test image first
test_images = glob(os.path.join(WALL_IMAGES_DIR, '*.png')) + glob(os.path.join(WALL_IMAGES_DIR, '*.jpg'))

if test_images:
    test_img = test_images[0]
    print(f"Testing with: {os.path.basename(test_img)}")
    
    seg_map, color_overlay, orig = segment_image(cnn_model, test_img)
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(orig)
    axes[0].set_title('Original Wall Image')
    axes[0].axis('off')
    
    axes[1].imshow(color_overlay)
    axes[1].set_title('Segmentation')
    axes[1].axis('off')
    
    blend = ((0.6 * orig) + (0.4 * color_overlay)).astype(np.uint8)
    axes[2].imshow(blend)
    axes[2].set_title('Overlay')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Show detected rooms
    unique_classes = np.unique(seg_map)
    print(f"\nDetected rooms: {[CLASS_NAMES[c] for c in unique_classes if c > 0]}")
else:
    print(f"No images found in {WALL_IMAGES_DIR}")

---
## 🎨 Part 2: LoRA Inference

Generate furnished room images using FLUX.1-dev with LoRA weights and Canny ControlNet.

In [ ]:
from diffusers import FluxControlPipeline
from diffusers.utils import load_image
from controlnet_aux import CannyDetector
import torch

# Initialize Canny detector (same as working notebooks)
canny_processor = CannyDetector()
print("✓ CannyDetector initialized")

# Room-specific prompts
ROOM_PROMPTS = {
    'living_room': "A top-down view of a modern living room floor plan with furniture including sofa, TV unit, coffee table, and rug. Clean architectural style, minimal design.",
    'bedroom': "A top-down view of a bedroom floor plan with bed, nightstands, wardrobe, and desk. Clean architectural style, minimal design.",
    'bathroom': "A top-down view of a bathroom floor plan with bathtub, toilet, sink, and shower. Clean architectural style, minimal design.",
    'hallway': "A top-down view of a hallway floor plan with minimal furniture. Clean architectural style, minimal design.",
    'toilet': "A top-down view of a small toilet room with toilet and sink. Clean architectural style, minimal design.",
    'empty_room': "A top-down view of an empty room floor plan. Clean architectural style, minimal design."
}

NEGATIVE_PROMPT = "blurry, distorted, low quality, text, watermark, 3D render, realistic photo"

print("✓ Prompts configured")

In [ ]:
class LoRAInferenceEngine:
    """Handles LoRA-based room generation using the same approach as individual inference notebooks."""
    
    def __init__(self, lora_weights_dict, base_model="black-forest-labs/FLUX.1-dev"):
        self.lora_weights = lora_weights_dict
        self.base_model = base_model
        self.pipeline = None
        self.current_lora = None
        self.canny_loaded = False
        
    def load_pipeline(self):
        """Load the base FLUX pipeline."""
        if self.pipeline is None:
            print("Loading FLUX.1-dev pipeline...")
            self.pipeline = FluxControlPipeline.from_pretrained(
                self.base_model,
                torch_dtype=torch.float16  # Use float16 like working notebooks
            )
            self.pipeline.to(device)
            
            # Enable memory optimization (like working notebooks)
            self.pipeline.enable_model_cpu_offload()
            print("✓ Pipeline loaded with memory optimization")
        return self.pipeline
    
    def load_lora_for_room(self, room_type):
        """Load LoRA weights for a specific room type + Canny LoRA."""
        # Load pipeline if not ready
        self.load_pipeline()
        
        # Map room type to LoRA path
        lora_path = self.lora_weights.get(room_type)
        if lora_path is None:
            # Fallback to living_room for unknown types
            lora_path = self.lora_weights.get('living_room')
            print(f"  ⚠ No LoRA for {room_type}, using living_room")
        
        if lora_path and os.path.exists(lora_path):
            # Unload previous LoRAs if switching room type
            if self.current_lora and self.current_lora != room_type:
                try:
                    self.pipeline.unload_lora_weights()
                except:
                    pass
                self.canny_loaded = False
            
            if self.current_lora != room_type:
                # Load room-specific LoRA
                self.pipeline.load_lora_weights(lora_path, adapter_name="room")
                
                # Load Canny ControlNet LoRA (CRITICAL - this is what working notebooks use!)
                self.pipeline.load_lora_weights(
                    "black-forest-labs/FLUX.1-Canny-dev-lora",
                    adapter_name="canny"
                )
                
                # Set adapter weights (like working notebooks)
                self.pipeline.set_adapters(
                    ["room", "canny"],
                    adapter_weights=[0.8, 0.7]
                )
                
                self.current_lora = room_type
                self.canny_loaded = True
                print(f"  ✓ Loaded LoRA: {room_type} + Canny adapter")
        else:
            print(f"  ⚠ LoRA path not found: {lora_path}")
    
    def generate_room(self, mask_path, room_type, output_path, num_inference_steps=30, guidance_scale=7.5):
        """
        Generate a furnished room image from a mask.
        Uses the SAME approach as working individual inference notebooks.
        
        Args:
            mask_path: Path to room mask image
            room_type: Type of room (living_room, bedroom, etc.)
            output_path: Where to save generated image
            
        Returns:
            Generated PIL Image
        """
        # Load LoRA for this room type
        self.load_lora_for_room(room_type)
        
        # Load mask image
        mask_img = load_image(mask_path)
        original_size = (mask_img.width, mask_img.height)
        
        # Use CannyDetector from controlnet_aux (SAME as working notebooks!)
        # This properly processes the image at 512x512 resolution
        control_image = canny_processor(
            mask_img,
            low_threshold=50,
            high_threshold=200,
            detect_resolution=512,
            image_resolution=512
        )
        
        # Get prompt for this room type
        prompt = ROOM_PROMPTS.get(room_type, ROOM_PROMPTS['empty_room'])
        
        # Generate (matching working notebook parameters)
        generator = torch.Generator(device="cuda").manual_seed(42)
        
        result = self.pipeline(
            prompt=prompt,
            control_image=control_image,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            generator=generator,
        ).images[0]
        
        # Resize back to original dimensions if needed
        if result.size != original_size:
            result = result.resize(original_size, Image.LANCZOS)
        
        # Save
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        result.save(output_path)
        
        return result


print("✓ LoRAInferenceEngine defined (matching working notebook approach)")

In [ ]:
# Initialize LoRA engine (will load on first use)
lora_engine = LoRAInferenceEngine(LORA_WEIGHTS)
print("✓ LoRA engine initialized (pipeline loads on first generation)")

---
## 🔗 Part 3: Integration (from integration notebook)

This section includes:
1. **FurnitureDetector**: YOLO-based furniture detection via Roboflow
2. **FurnitureMapper**: Maps detected furniture to room types with STRICT validation
3. **EnhancedFurniturePlacer**: Rule-based furniture placement with polygon validation
4. **Floor Plan Combination**: Merges all generated rooms into final output

In [ ]:
# ============================================================
# FURNITURE DETECTOR (from integration notebook)
# ============================================================

class DetectedFurniture:
    """Represents a piece of furniture detected by YOLO"""
    def __init__(self, furniture_type: str, bbox: Tuple[int, int, int, int],
                 polygon_points: List[Tuple[int, int]], confidence: float,
                 center: Tuple[int, int], angle: float = 0):
        self.furniture_type = furniture_type
        self.bbox = bbox  # (x, y, w, h)
        self.polygon_points = polygon_points
        self.confidence = confidence
        self.center = center
        self.angle = angle

    def __repr__(self):
        return f"DetectedFurniture({self.furniture_type}, conf={self.confidence:.2f}, center={self.center})"


class FurnitureDetector:
    """Detects furniture in floor plan images using YOLO via Roboflow"""

    def __init__(self, api_key: str = ROBOFLOW_API_KEY, model_id: str = ROBOFLOW_MODEL_ID):
        self.client = InferenceHTTPClient(
            api_url="https://serverless.roboflow.com",
            api_key=api_key
        )
        self.model_id = model_id

    def detect_furniture(self, image_path: str, confidence_threshold: float = 0.3) -> List[DetectedFurniture]:
        """Detect furniture in an image using YOLO model."""
        try:
            result = self.client.infer(image_path, model_id=self.model_id)
        except Exception as e:
            print(f"  ⚠ Detection failed for {os.path.basename(image_path)}: {e}")
            return []

        detected_furniture = []

        for prediction in result.get('predictions', []):
            confidence = prediction.get('confidence', 0)
            if confidence < confidence_threshold:
                continue

            furniture_type = prediction.get('class', 'unknown')
            center_x = int(prediction.get('x', 0))
            center_y = int(prediction.get('y', 0))

            # Get polygon points if available
            polygon_points = []
            if 'points' in prediction:
                polygon_points = [(int(p['x']), int(p['y'])) for p in prediction['points']]

            # Calculate bounding box
            if polygon_points:
                xs = [p[0] for p in polygon_points]
                ys = [p[1] for p in polygon_points]
                x_min, x_max = min(xs), max(xs)
                y_min, y_max = min(ys), max(ys)
                bbox = (x_min, y_min, x_max - x_min, y_max - y_min)
            else:
                width = int(prediction.get('width', 50))
                height = int(prediction.get('height', 50))
                bbox = (center_x - width//2, center_y - height//2, width, height)

            # Calculate orientation angle from polygon
            angle = 0
            if polygon_points and len(polygon_points) >= 4:
                pts = np.array(polygon_points, dtype=np.float32)
                rect = cv2.minAreaRect(pts)
                angle = rect[2]
                if angle < -45:
                    angle = 90 + angle
                elif angle > 45:
                    angle = angle - 90

            detected = DetectedFurniture(
                furniture_type=furniture_type,
                bbox=bbox,
                polygon_points=polygon_points,
                confidence=confidence,
                center=(center_x, center_y),
                angle=angle
            )
            detected_furniture.append(detected)

        return detected_furniture

    def detect_in_multiple_images(self, image_paths: List[str],
                                  confidence_threshold: float = 0.3) -> Dict[str, List[DetectedFurniture]]:
        """Detect furniture in multiple images."""
        detections = {}

        for img_path in image_paths:
            print(f"  Detecting in: {os.path.basename(img_path)}")
            furniture = self.detect_furniture(img_path, confidence_threshold)
            detections[img_path] = furniture

            if furniture:
                furniture_types = [f.furniture_type for f in furniture]
                print(f"    Found {len(furniture)} items: {', '.join(furniture_types)}")
            else:
                print(f"    No furniture detected")

        return detections


print("✓ FurnitureDetector class defined")

In [ ]:
# ============================================================
# FURNITURE MAPPER WITH STRICT ROOM VALIDATION
# ============================================================

# CORRECTED: Sink ONLY in bathroom/toilet (NOT living_room!)
CORRECTED_FURNITURE_ROOM_MAP = {
    "living_room": ["Lsofa", "sofa", "tv", "table", "diningtable", "kitchen", "stove"],
    "bedroom_master": ["bed", "bedside", "study", "table"],
    "bedroom_guest": ["singlebed", "bedside", "study", "table"],
    "bedroom": ["singlebed", "bed", "bedside", "study", "table"],
    "bathroom": ["bathtub", "shower", "sink", "commode"],
    "toilet": ["commode", "sink"],
    "hallway": [],
    "empty_room": [],
}

# Maximum allowed instances per furniture type
MAX_FURNITURE_PER_ROOM = {
    "bed": 1, "singlebed": 1, "Lsofa": 1, "sofa": 1, "tv": 1,
    "diningtable": 1, "table": 1, "kitchen": 1, "stove": 1,
    "bathtub": 1, "shower": 1, "sink": 1, "commode": 1,
    "bedside": 2, "study": 1,
}


class FurnitureMapper:
    """Maps YOLO-detected furniture to floor_planrules format with STRICT validation"""

    YOLO_TO_FLOORPLAN_MAP = {
        'bed': 'bed', 'single_bed': 'singlebed', 'singlebed': 'singlebed',
        'sofa': 'sofa', 'l_sofa': 'Lsofa', 'L-sofa': 'Lsofa', 'Lsofa': 'Lsofa',
        'tv': 'tv', 'table': 'table', 'dining_table': 'diningtable',
        'diningtable': 'diningtable', 'bathtub': 'bathtub', 'shower': 'shower',
        'sink': 'sink', 'commode': 'commode', 'toilet': 'commode',
        'bedside': 'bedside', 'study': 'study', 'desk': 'study',
        'kitchen': 'kitchen', 'stove': 'stove', 'door': 'door', 'chair': 'chair',
    }

    def __init__(self, pixel_to_meter_ratio: float = 0.1, furniture_prototypes=None):
        self.pixel_to_meter_ratio = pixel_to_meter_ratio
        self.furniture_prototypes = furniture_prototypes or {}

    def _get_room_base_type(self, room_name: str) -> str:
        """Extract base room type (bedroom_1 -> bedroom)"""
        room_name_lower = room_name.lower()
        compound_types = ['living_room', 'bedroom_master', 'bedroom_guest', 'empty_room']
        for compound in compound_types:
            if room_name_lower.startswith(compound):
                return compound
        parts = room_name_lower.split('_')
        if len(parts) >= 2 and parts[-1].isdigit():
            return '_'.join(parts[:-1])
        return room_name_lower

    def furniture_belongs_to_room(self, furniture_name: str, room_name: str) -> bool:
        """STRICT check if furniture belongs in room type"""
        if furniture_name is None:
            return False
        room_type = self._get_room_base_type(room_name)
        valid_furniture = CORRECTED_FURNITURE_ROOM_MAP.get(room_type, [])
        furniture_name_lower = furniture_name.lower()
        for valid_type in valid_furniture:
            if furniture_name_lower == valid_type.lower() or \
               furniture_name_lower in valid_type.lower() or \
               valid_type.lower() in furniture_name_lower:
                return True
        return False

    def map_furniture_type(self, yolo_type: str) -> str:
        """Map YOLO type to floor_planrules type"""
        yolo_clean = yolo_type.lower().strip().replace(' ', '_').replace('-', '')
        if yolo_clean in self.YOLO_TO_FLOORPLAN_MAP:
            return self.YOLO_TO_FLOORPLAN_MAP[yolo_clean]
        for yolo_key, floorplan_type in self.YOLO_TO_FLOORPLAN_MAP.items():
            if yolo_key in yolo_clean or yolo_clean in yolo_key:
                return floorplan_type
        return None

    def convert_to_furniture_object(self, detected: DetectedFurniture) -> Furniture:
        """Convert DetectedFurniture to Furniture object"""
        mapped_type = self.map_furniture_type(detected.furniture_type)
        if mapped_type is None:
            return None

        # Get dimensions from JSON prototypes
        width_m, height_m = None, None
        if mapped_type in self.furniture_prototypes and self.furniture_prototypes[mapped_type]:
            proto = self.furniture_prototypes[mapped_type][0]
            width_m, height_m = proto.width_m, proto.height_m

        if width_m is None:
            width_m = detected.bbox[2] * self.pixel_to_meter_ratio
            height_m = detected.bbox[3] * self.pixel_to_meter_ratio

        furniture = Furniture(
            name=mapped_type,
            width_m=width_m,
            height_m=height_m,
            original_filename=f"{detected.furniture_type}.png"
        )
        furniture.detected_position_px = detected.center
        furniture.detected_angle = detected.angle
        furniture.confidence = detected.confidence
        furniture.is_from_detection = True
        furniture.width_px = width_m / self.pixel_to_meter_ratio
        furniture.height_px = height_m / self.pixel_to_meter_ratio
        return furniture

    def group_by_room_with_validation(self, detections_by_image: Dict[str, List[DetectedFurniture]]) -> tuple:
        """Group and STRICTLY validate furniture by room"""
        valid_furniture = {}
        invalid_furniture = {}

        for img_path, detected_list in detections_by_image.items():
            filename = os.path.basename(img_path)
            # Extract room instance from filename
            name = filename.replace('.png', '').replace('.jpg', '').replace('_extracted', '').replace('_generated', '')
            parts = name.split('_')
            if 'living' in parts and 'room' in parts:
                idx = parts.index('room')
                if idx + 1 < len(parts) and parts[idx + 1].isdigit():
                    room_instance = f"living_room_{parts[idx + 1]}"
                else:
                    room_instance = "living_room"
            elif len(parts) >= 2 and parts[-1].isdigit():
                room_instance = '_'.join(parts[:-1]) + '_' + parts[-1]
            else:
                room_instance = parts[0] if parts else 'unknown'

            print(f"\nProcessing {filename} -> Room: {room_instance}")

            for detected in detected_list:
                furniture = self.convert_to_furniture_object(detected)
                if furniture is None:
                    print(f"    ✗ Unknown type: {detected.furniture_type}")
                    continue

                if self.furniture_belongs_to_room(furniture.name, room_instance):
                    if room_instance not in valid_furniture:
                        valid_furniture[room_instance] = []
                    valid_furniture[room_instance].append(furniture)
                    print(f"    ✓ {furniture.name} is valid")
                else:
                    if room_instance not in invalid_furniture:
                        invalid_furniture[room_instance] = []
                    invalid_furniture[room_instance].append(furniture)
                    print(f"    ✗ REJECTED: {furniture.name} does NOT belong in {room_instance}")

        return valid_furniture, invalid_furniture


print("✓ FurnitureMapper class defined with STRICT room validation")
print("  - Sink ONLY allowed in bathroom/toilet")

In [ ]:
# ============================================================
# ENHANCED FURNITURE PLACER (Rule-Based Placement)
# ============================================================

class EnhancedFurniturePlacer(FurniturePlacer):
    """
    Enhanced placer combining detected furniture with rule-based placement.
    
    Features:
    - Uses detected positions as hints with validation
    - Strict boundary checking (furniture must be FULLY inside room)
    - No overlaps - maintains global placed furniture list
    - Fills empty rooms with appropriate prototypes
    - Limits each furniture type to MAX instances per room
    """

    def __init__(self, rooms_info, furniture_prototypes, furniture_map, pixel_to_meter_ratio,
                 detected_furniture_by_room=None, use_detected_positions=True,
                 position_tolerance=50, debug=False, min_wall_distance=5):
        # Store original prototypes
        self.original_prototypes = {}
        for name, proto_list in furniture_prototypes.items():
            self.original_prototypes[name] = []
            for proto in proto_list:
                proto_copy = Furniture(name=proto.name, width_m=proto.width_m,
                                       height_m=proto.height_m, original_filename=proto.original_filename)
                proto_copy.width_px = proto.width_m / pixel_to_meter_ratio
                proto_copy.height_px = proto.height_m / pixel_to_meter_ratio
                self.original_prototypes[name].append(proto_copy)

        super().__init__(rooms_info, furniture_prototypes, CORRECTED_FURNITURE_ROOM_MAP, pixel_to_meter_ratio, debug)
        self.detected_furniture_by_room = detected_furniture_by_room or {}
        self.use_detected_positions = use_detected_positions
        self.position_tolerance = position_tolerance
        self.min_wall_distance = min_wall_distance
        self.global_placed_polygons = []

    def _get_furniture_polygon(self, furniture, center_pos, angle):
        """Creates a shapely Polygon for furniture item."""
        w, h = furniture.width_px, furniture.height_px
        corners = [(-w/2, -h/2), (w/2, -h/2), (w/2, h/2), (-w/2, h/2)]
        poly = Polygon(corners)
        poly = shapely_rotate(poly, angle, origin='center', use_radians=False)
        poly = shapely_translate(poly, xoff=center_pos[0], yoff=center_pos[1])
        return poly

    def _check_valid_placement(self, furniture_poly, room_polygon, placed_polygons, buffer_margin=2.0):
        """STRICT placement validation."""
        for placed_poly in placed_polygons:
            if furniture_poly.intersects(placed_poly):
                return False
        for global_poly in self.global_placed_polygons:
            if furniture_poly.intersects(global_poly):
                return False
        if room_polygon.contains(furniture_poly):
            return True
        if room_polygon.buffer(buffer_margin).contains(furniture_poly):
            return True
        return False

    def _create_fresh_prototype(self, furniture_type):
        """Create a FRESH COPY of a furniture prototype."""
        if furniture_type in self.original_prototypes and self.original_prototypes[furniture_type]:
            proto = self.original_prototypes[furniture_type][0]
            copy = Furniture(name=proto.name, width_m=proto.width_m,
                           height_m=proto.height_m, original_filename=proto.original_filename)
            copy.width_px = proto.width_px
            copy.height_px = proto.height_px
            copy.is_from_detection = False
            return copy
        return None

    def _place_with_wall_rules(self, furniture, room_data, room_polygon, placed_polygons):
        """Place furniture using wall-based rules."""
        walls = self._get_walls(room_data['bounding_box'])
        walls.sort(key=lambda w: np.linalg.norm(w[1] - w[0]), reverse=True)

        for wall in walls:
            wall_vec = wall[1] - wall[0]
            wall_len = np.linalg.norm(wall_vec)
            wall_angle = np.rad2deg(np.arctan2(wall_vec[1], wall_vec[0]))

            if furniture.width_px > wall_len * 0.9:
                continue

            for t in [0.5, 0.3, 0.7, 0.2, 0.8]:
                pos = wall[0] + wall_vec * t
                normal = self._get_wall_normal(pos, room_polygon, wall_vec)
                pos_adjusted = pos + normal * (furniture.height_px / 2 + self.min_wall_distance)

                poly = self._get_furniture_polygon(furniture, pos_adjusted, wall_angle)
                if self._check_valid_placement(poly, room_polygon, placed_polygons):
                    furniture.position_px = (int(pos_adjusted[0]), int(pos_adjusted[1]))
                    furniture.angle = wall_angle
                    return True, poly
        return False, None

    def _get_wall_normal(self, point, room_polygon, wall_vec):
        """Gets normal vector pointing into room from wall."""
        normal = np.array([-wall_vec[1], wall_vec[0]])
        norm_val = np.linalg.norm(normal)
        if norm_val > 0:
            normal = normal / norm_val
        centroid = np.array(room_polygon.centroid.coords[0])
        if np.dot(normal, centroid - point) < 0:
            normal = -normal
        return normal


print("✓ EnhancedFurniturePlacer defined with rule-based placement")

In [ ]:
# ============================================================
# FLOOR PLAN INTEGRATION FUNCTIONS
# ============================================================

def load_all_room_images(room_folders):
    """Load all extracted room images from each room type folder."""
    room_images = {}
    for room_type, folder_path in room_folders.items():
        if not os.path.exists(folder_path):
            print(f"Warning: {room_type} folder not found at {folder_path}")
            continue

        image_files = glob(os.path.join(folder_path, "*.png"))
        room_instances = {}

        for img_path in image_files:
            filename = os.path.basename(img_path)
            parts = filename.split('_')
            if len(parts) >= 2 and parts[1].isdigit():
                room_instance = f"{room_type}_{parts[1]}"
            else:
                room_instance = room_type

            img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
            if img is not None:
                if len(img.shape) == 3 and img.shape[2] == 4:
                    img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGBA)
                elif len(img.shape) == 3:
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                if room_instance not in room_instances:
                    room_instances[room_instance] = []
                room_instances[room_instance].append((img_path, img))

        for room_instance, images in room_instances.items():
            room_images[room_instance] = images
            print(f"Loaded {len(images)} images for {room_instance}")

    return room_images


def create_floor_plan_combination(room_images, base_size):
    """Create floor plan by overlaying room images on canvas."""
    canvas = np.ones((base_size[0], base_size[1], 3), dtype=np.uint8) * 255
    selected_rooms = {}

    for room_type, images in room_images.items():
        if len(images) == 0:
            continue
        img_path, room_img = random.choice(images)
        if room_img.shape[:2] != (base_size[0], base_size[1]):
            room_img = cv2.resize(room_img, (base_size[1], base_size[0]), interpolation=cv2.INTER_AREA)

        if room_img.shape[2] == 4:
            room_rgb = room_img[:, :, :3]
            alpha = room_img[:, :, 3:4] / 255.0
            canvas = (alpha * room_rgb + (1 - alpha) * canvas).astype(np.uint8)
        else:
            mask = ~np.all(room_img == 255, axis=2)
            canvas[mask] = room_img[mask]
        selected_rooms[room_type] = os.path.basename(img_path)

    return canvas, selected_rooms


def generate_floor_plan_with_furniture(
    lora_room_images,
    segmented_mask_path,
    furniture_json_path,
    output_folder,
    pixel_to_meter_ratio=0.1,
    confidence_threshold=0.3,
    debug=True
):
    """
    Complete integration workflow:
    LoRA output -> YOLO detection -> STRICT Validation -> Rule-Based Placement
    """
    print("="*80)
    print("FLOOR PLAN FURNITURE PLACEMENT")
    print("="*80)

    # Step 1: Detect furniture from LoRA outputs
    print("\n📍 STEP 1: Detecting furniture using YOLO")
    detector = FurnitureDetector()
    detections = detector.detect_in_multiple_images(lora_room_images, confidence_threshold)
    total_detected = sum(len(d) for d in detections.values())
    print(f"✓ Total furniture detected: {total_detected}")

    # Step 2: Load furniture prototypes
    print("\n📍 STEP 2: Loading furniture prototypes")
    furniture_prototypes = load_furniture_prototypes(furniture_json_path)
    for name in furniture_prototypes:
        for f_item in furniture_prototypes[name]:
            f_item.width_px = f_item.width_m / pixel_to_meter_ratio
            f_item.height_px = f_item.height_m / pixel_to_meter_ratio
    print(f"✓ Loaded {sum(len(items) for items in furniture_prototypes.values())} prototypes")

    # Step 3: Validate furniture-room mapping
    print("\n📍 STEP 3: STRICT validation - removing wrong furniture")
    mapper = FurnitureMapper(pixel_to_meter_ratio, furniture_prototypes)
    valid_furniture, invalid_furniture = mapper.group_by_room_with_validation(detections)
    print(f"✓ Valid: {sum(len(f) for f in valid_furniture.values())}")
    print(f"✗ Rejected: {sum(len(f) for f in invalid_furniture.values())}")

    # Step 4: Analyze room structure
    print("\n📍 STEP 4: Analyzing room structure")
    segmented_image = cv2.imread(segmented_mask_path)
    segmented_image = cv2.cvtColor(segmented_image, cv2.COLOR_BGR2RGB)
    rooms_info = extract_room_dimensions(segmented_image, color_maps, room_types, pixel_to_meter_ratio)
    print(f"✓ Found {len(rooms_info)} rooms")

    # Step 5: Place furniture
    print("\n📍 STEP 5: Rule-based furniture placement")
    placer = EnhancedFurniturePlacer(
        rooms_info=rooms_info,
        furniture_prototypes=furniture_prototypes,
        furniture_map=CORRECTED_FURNITURE_ROOM_MAP,
        pixel_to_meter_ratio=pixel_to_meter_ratio,
        detected_furniture_by_room=valid_furniture,
        debug=debug
    )
    placed_furniture = placer.place_all()

    # Step 6: Render final floor plan
    print("\n📍 STEP 6: Rendering final floor plan")
    final_floor_plan = placer.render_floor_plan(segmented_image)

    # Save output
    os.makedirs(output_folder, exist_ok=True)
    output_path = os.path.join(output_folder, "final_floor_plan.png")
    cv2.imwrite(output_path, cv2.cvtColor(final_floor_plan, cv2.COLOR_RGB2BGR))
    print(f"✓ Saved: {output_path}")

    return final_floor_plan, placed_furniture


print("✓ Floor plan integration functions defined")

In [ ]:
def combine_room_images(generated_rooms, original_size, seg_map):
    """
    Combine generated room images into a single floor plan.
    
    Args:
        generated_rooms: Dict mapping room_type -> list of (image_path, bbox)
        original_size: (height, width) of original image
        seg_map: Segmentation map for positioning
        
    Returns:
        Combined floor plan image
    """
    H, W = original_size
    canvas = np.ones((H, W, 3), dtype=np.uint8) * 255  # White background
    
    for room_type, room_list in generated_rooms.items():
        for room_info in room_list:
            img_path = room_info['generated_path']
            bbox = room_info['bbox']
            x, y, w, h = bbox
            
            # Load generated image
            room_img = cv2.imread(img_path)
            if room_img is None:
                continue
            room_img = cv2.cvtColor(room_img, cv2.COLOR_BGR2RGB)
            
            # Resize to fit bbox
            room_img = cv2.resize(room_img, (w, h))
            
            # Get mask for this room
            class_idx = list(CLASS_TO_ROOM.keys())[list(CLASS_TO_ROOM.values()).index(room_type)]
            room_mask = (seg_map[y:y+h, x:x+w] == class_idx).astype(np.uint8)
            
            # Apply mask and blend onto canvas
            for c in range(3):
                canvas[y:y+h, x:x+w, c] = np.where(
                    room_mask,
                    room_img[:, :, c],
                    canvas[y:y+h, x:x+w, c]
                )
    
    return canvas


def overlay_walls(floor_plan, wall_image, seg_map):
    """
    Overlay wall boundaries from original image.
    """
    # Find wall pixels (background in segmentation)
    wall_mask = (seg_map == 0)
    
    # Convert wall image to grayscale to detect dark lines
    wall_gray = cv2.cvtColor(wall_image, cv2.COLOR_RGB2GRAY)
    _, wall_binary = cv2.threshold(wall_gray, 200, 255, cv2.THRESH_BINARY_INV)
    
    # Overlay walls
    result = floor_plan.copy()
    result[wall_binary > 0] = [0, 0, 0]  # Black walls
    
    return result


print("✓ Integration functions defined")

---
## 🚀 Part 4: Full Pipeline Execution

In [ ]:
def run_full_pipeline(wall_image_path, output_prefix, use_furniture_detection=True):
    """
    Run the complete pipeline on a single wall image.
    
    Pipeline Steps:
    1. CNN Segmentation → Room masks
    2. Room Extraction → Individual room regions
    3. LoRA Generation → Furnished room images
    4. Furniture Detection → YOLO detects furniture in LoRA output
    5. Furniture Validation → STRICT room-furniture mapping
    6. Rule-Based Placement → EnhancedFurniturePlacer positions furniture
    7. Integration → Final floor plan
    
    Args:
        wall_image_path: Path to input wall/boundary image
        output_prefix: Prefix for output files
        use_furniture_detection: Whether to run YOLO furniture detection (default True)
        
    Returns:
        final_floor_plan: Combined floor plan image
        metadata: Pipeline metadata dict
    """
    print(f"\n{'='*60}")
    print(f"🏠 FULL PIPELINE: {os.path.basename(wall_image_path)}")
    print(f"{'='*60}")
    
    metadata = {'input': wall_image_path, 'rooms': {}, 'furniture': {}}
    
    # ---- Step 1: CNN Segmentation ----
    print("\n📍 Step 1: CNN Segmentation")
    seg_map, color_overlay, orig = segment_image(cnn_model, wall_image_path)
    
    # Save segmentation
    seg_path = os.path.join(SEGMENTATION_OUTPUT, f"{output_prefix}_segmentation.png")
    cv2.imwrite(seg_path, cv2.cvtColor(color_overlay, cv2.COLOR_RGB2BGR))
    metadata['segmentation_path'] = seg_path
    print(f"  ✓ Segmentation saved: {seg_path}")
    
    # ---- Step 2: Extract Room Masks ----
    print("\n📍 Step 2: Extracting Room Masks")
    room_masks = extract_room_masks(seg_map, orig, EXTRACTION_OUTPUT, output_prefix)
    
    for room_type, masks in room_masks.items():
        print(f"  ✓ {room_type}: {len(masks)} instance(s)")
    
    # ---- Step 3: LoRA Generation ----
    print("\n📍 Step 3: LoRA Room Generation")
    generated_rooms = {}
    generated_image_paths = []
    
    for room_type, mask_list in room_masks.items():
        generated_rooms[room_type] = []
        
        for mask_info in mask_list:
            mask_path = mask_info['path']
            instance = mask_info['instance']
            
            output_name = f"{output_prefix}_{room_type}_{instance}_generated.png"
            output_path = os.path.join(LORA_OUTPUT, room_type, output_name)
            
            print(f"  Generating {room_type} #{instance}...")
            
            try:
                generated_img = lora_engine.generate_room(
                    mask_path, room_type, output_path
                )
                
                generated_rooms[room_type].append({
                    'generated_path': output_path,
                    'bbox': mask_info['bbox'],
                    'mask_path': mask_path
                })
                generated_image_paths.append(output_path)
                print(f"    ✓ Saved: {output_path}")
                
            except Exception as e:
                print(f"    ✗ Failed: {e}")
    
    metadata['rooms'] = generated_rooms
    
    # ---- Step 4: Furniture Detection (YOLO) ----
    if use_furniture_detection and generated_image_paths:
        print("\n📍 Step 4: YOLO Furniture Detection")
        detector = FurnitureDetector()
        detections = detector.detect_in_multiple_images(generated_image_paths, confidence_threshold=0.3)
        total_detected = sum(len(d) for d in detections.values())
        print(f"  ✓ Total furniture detected: {total_detected}")
        
        # ---- Step 5: Furniture Validation ----
        print("\n📍 Step 5: STRICT Furniture Validation")
        furniture_prototypes = load_furniture_prototypes(FURNITURE_JSON_PATH)
        for name in furniture_prototypes:
            for f_item in furniture_prototypes[name]:
                f_item.width_px = f_item.width_m / 0.1  # pixel_to_meter_ratio
                f_item.height_px = f_item.height_m / 0.1
        
        mapper = FurnitureMapper(pixel_to_meter_ratio=0.1, furniture_prototypes=furniture_prototypes)
        valid_furniture, invalid_furniture = mapper.group_by_room_with_validation(detections)
        
        valid_count = sum(len(f) for f in valid_furniture.values())
        invalid_count = sum(len(f) for f in invalid_furniture.values())
        print(f"  ✓ Valid furniture: {valid_count}")
        print(f"  ✗ Rejected (wrong room): {invalid_count}")
        
        metadata['furniture'] = {
            'valid_count': valid_count,
            'invalid_count': invalid_count,
            'by_room': {k: len(v) for k, v in valid_furniture.items()}
        }
        
        # ---- Step 6: Rule-Based Placement ----
        print("\n📍 Step 6: Rule-Based Furniture Placement")
        rooms_info = extract_room_dimensions(color_overlay, color_maps, room_types, 0.1)
        print(f"  ✓ Extracted {len(rooms_info)} rooms for placement")
        
        placer = EnhancedFurniturePlacer(
            rooms_info=rooms_info,
            furniture_prototypes=furniture_prototypes,
            furniture_map=CORRECTED_FURNITURE_ROOM_MAP,
            pixel_to_meter_ratio=0.1,
            detected_furniture_by_room=valid_furniture,
            debug=False
        )
        placed_furniture = placer.place_all()
        
        total_placed = sum(len(f) for f in placed_furniture.values())
        print(f"  ✓ Placed {total_placed} furniture items")
    
    # ---- Step 7: Integration ----
    print("\n📍 Step 7: Combining into Final Floor Plan")
    
    original_size = orig.shape[:2]
    combined = combine_room_images(generated_rooms, original_size, seg_map)
    final_floor_plan = overlay_walls(combined, orig, seg_map)
    
    # Save final result
    final_path = os.path.join(FINAL_OUTPUT, f"{output_prefix}_final.png")
    cv2.imwrite(final_path, cv2.cvtColor(final_floor_plan, cv2.COLOR_RGB2BGR))
    print(f"  ✓ Final floor plan saved: {final_path}")
    
    metadata['output'] = final_path
    
    # Save metadata
    metadata_path = os.path.join(FINAL_OUTPUT, f"{output_prefix}_metadata.json")
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2, default=str)
    
    print(f"\n{'='*60}")
    print("✅ PIPELINE COMPLETE!")
    print(f"{'='*60}")
    
    return final_floor_plan, metadata


print("✓ Full pipeline function defined with integration")

In [ ]:
# ============================================================
# RUN THE PIPELINE ON A SINGLE IMAGE
# ============================================================

# Get list of wall images
wall_images = glob(os.path.join(WALL_IMAGES_DIR, '*.png')) + glob(os.path.join(WALL_IMAGES_DIR, '*.jpg'))
print(f"Found {len(wall_images)} wall images")

if wall_images:
    # Process first image as demo
    test_image = wall_images[0]
    base_name = os.path.splitext(os.path.basename(test_image))[0]
    
    final_plan, metadata = run_full_pipeline(test_image, base_name)
    
    # Visualize result
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    orig_img = cv2.cvtColor(cv2.imread(test_image), cv2.COLOR_BGR2RGB)
    plt.imshow(orig_img)
    plt.title('Input: Wall Image')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(final_plan)
    plt.title('Output: Furnished Floor Plan')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print(f"No wall images found in {WALL_IMAGES_DIR}")

---
## 📊 Batch Processing

In [ ]:
def run_batch_pipeline(wall_images_dir, max_images=None):
    """
    Process multiple wall images through the full pipeline.
    
    Args:
        wall_images_dir: Directory containing wall images
        max_images: Maximum number of images to process (None = all)
        
    Returns:
        results: List of (final_plan, metadata) tuples
    """
    wall_images = glob(os.path.join(wall_images_dir, '*.png')) + glob(os.path.join(wall_images_dir, '*.jpg'))
    
    if max_images:
        wall_images = wall_images[:max_images]
    
    print(f"\n🚀 Processing {len(wall_images)} images...\n")
    
    results = []
    for i, img_path in enumerate(tqdm(wall_images, desc="Processing")):
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        output_prefix = f"{i+1:03d}_{base_name}"
        
        try:
            final_plan, metadata = run_full_pipeline(img_path, output_prefix)
            results.append((final_plan, metadata))
        except Exception as e:
            print(f"\n✗ Failed on {img_path}: {e}")
            continue
    
    print(f"\n\n{'='*60}")
    print(f"✅ Batch processing complete: {len(results)}/{len(wall_images)} successful")
    print(f"{'='*60}")
    
    return results


print("✓ Batch processing function defined")

In [ ]:
# Uncomment to run batch processing
# results = run_batch_pipeline(WALL_IMAGES_DIR, max_images=5)

---
## 📈 Pipeline Summary

This notebook implements the **complete end-to-end floor plan generation pipeline**:

| Stage | Component | Description |
|-------|-----------|-------------|
| 1 | **CNN Segmentation** | DeepLabV3+ extracts room masks from wall images |
| 2 | **Room Extraction** | Separates individual room regions with bounding boxes |
| 3 | **LoRA Generation** | FLUX.1-dev + LoRA generates furnished room images |
| 4 | **Furniture Detection** | YOLO (Roboflow) detects furniture in generated images |
| 5 | **Furniture Validation** | STRICT room-furniture mapping (sink only in bathroom!) |
| 6 | **Rule-Based Placement** | EnhancedFurniturePlacer positions furniture with polygon validation |
| 7 | **Integration** | Combines all rooms into final floor plan |

### Key Metrics Achieved:
- **Segmentation**: 97.1% mIoU, 99.1% pixel accuracy (DeepLabV3+)
- **Detection**: 87.0% mAP@50, 80.4% precision, 88.4% recall (YOLOv8)

### Integration Features (from integration notebook):
- ✅ CORRECTED furniture-room mapping (sink NOT in living room)
- ✅ YOLO furniture detection via Roboflow API
- ✅ STRICT validation removes hallucinated furniture
- ✅ Rule-based placement with wall alignment
- ✅ Polygon-based collision detection
- ✅ Fresh prototype copies (never runs out of furniture)